# Mapas AROME centrados en Chamartín

Prototipo independiente para explorar las previsiones de **temperatura a 2 m** y **precipitación horaria** de AROME publicadas por Meteociel. Los mapas ocupan un cuadrado de unos **100 × 100 km** (≈10.000 km²), centrado en Chamartín, sobre cartografía topográfica de OpenTopoMap.

El color y las isolíneas se dibujan con transparencia sobre el mapa: carreteras, ríos y relieve quedan visibles, pero secundarios. La primera ejecución descarga la rejilla y la guarda en `datos_arome_madrid/`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from io import BytesIO
import base64

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from bs4 import BeautifulSoup
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Chamartín y área de visualización de aproximadamente 10.000 km².
CHAMARTIN_LAT, CHAMARTIN_LON = 40.4623, -3.7004
HALF_SIDE_KM = 50
LAT_HALF_SPAN = HALF_SIDE_KM / 111.32
LON_HALF_SPAN = HALF_SIDE_KM / (111.32 * np.cos(np.deg2rad(CHAMARTIN_LAT)))
SOUTH, NORTH = CHAMARTIN_LAT - LAT_HALF_SPAN, CHAMARTIN_LAT + LAT_HALF_SPAN
WEST, EAST = CHAMARTIN_LON - LON_HALF_SPAN, CHAMARTIN_LON + LON_HALF_SPAN

# 5 km ofrece isolíneas suaves y limita el número de consultas a Meteociel.
GRID_STEP = 0.05
MAX_WORKERS = 3
REQUEST_PAUSE = 0.15
CACHE_DIR = Path('datos_arome_madrid')
CACHE_DIR.mkdir(exist_ok=True)
USER_AGENT = {'User-Agent': 'MeteoDash Madrid map prototype (personal use)'}
print(f'Área: {SOUTH:.3f}–{NORTH:.3f} N, {WEST:.3f}–{EAST:.3f} E')

In [ ]:
def square_points(step=GRID_STEP):
    lats = np.arange(np.floor(SOUTH / step) * step, NORTH + step / 2, step)
    lons = np.arange(np.floor(WEST / step) * step, EAST + step / 2, step)
    return [(round(lat, 4), round(lon, 4)) for lat in lats for lon in lons]

print(f'Rejilla: {len(square_points())} puntos de previsión')

In [ ]:
def parse_meteociel(url):
    response = requests.get(url, timeout=45, headers=USER_AGENT)
    response.raise_for_status()
    table = BeautifulSoup(response.text, 'html.parser').find('table', {'class': 'gefs'})
    if table is None:
        raise ValueError('Meteociel no devolvió la tabla AROME esperada')
    rows = table.find_all('tr')
    headers = [cell.get_text(strip=True) for cell in rows[0].find_all('td')]
    values = [[cell.get_text(strip=True) for cell in row.find_all('td')] for row in rows[1:]]
    frame = pd.DataFrame(values, columns=headers)
    frame['Date'] = pd.to_datetime(frame['Date'], utc=True).dt.tz_convert('Europe/Madrid')
    frame = frame.set_index('Date').drop(columns='Ech.').apply(pd.to_numeric, errors='coerce')
    # La app existente usa el promedio del conjunto para sus resúmenes.
    return frame.mean(axis=1).rename('value')

def arome_url(lat, lon, mode, run=None):
    url = ('https://www.meteociel.fr/modeles/pe-arome_table.php?'
           f'x=0&y=0&lat={lat:.4f}&lon={lon:.4f}&mode={mode}&sort=0')
    return f'{url}&run={run}' if run is not None else url

def latest_run():
    # Se escoge la pasada que llega más lejos, igual que el enfoque de la app actual.
    candidate_runs = (3, 9, 15, 21)
    reference = (40.4168, -3.7038)
    candidates = {}
    for run in candidate_runs:
        try:
            series = parse_meteociel(arome_url(*reference, mode=8, run=run))
            candidates[run] = series.index.max()
        except Exception as error:
            print(f'Pasada {run:02d}Z no disponible: {error}')
    if not candidates:
        raise RuntimeError('No se pudo determinar una pasada AROME disponible')
    return max(candidates, key=candidates.get)

RUN = latest_run()
print(f'Pasada AROME seleccionada: {RUN:02d}Z')

In [ ]:
def fetch_point(lat, lon, mode, run):
    last_error = None
    for attempt in range(3):
        try:
            series = parse_meteociel(arome_url(lat, lon, mode, run))
            time.sleep(REQUEST_PAUSE)
            return pd.DataFrame({'time': series.index, 'lat': lat, 'lon': lon, 'value': series.values})
        except Exception as error:
            last_error = error
            time.sleep(1.5 * (attempt + 1))
    raise last_error

def download_field(mode, name, run=RUN):
    cache_file = CACHE_DIR / f'arome_{name}_run_{run:02d}_square_100km_step_{GRID_STEP:.3f}.pkl'
    if cache_file.exists():
        print(f'Reutilizando caché: {cache_file.name}')
        return pd.read_pickle(cache_file)

    points = square_points()
    results, failures = [], []
    print(f'Descargando {name}: {len(points)} puntos, pasada {run:02d}Z…')
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(fetch_point, lat, lon, mode, run): (lat, lon) for lat, lon in points}
        for number, future in enumerate(as_completed(futures), start=1):
            try:
                results.append(future.result())
            except Exception as error:
                failures.append((*futures[future], str(error)))
            if number % 25 == 0 or number == len(points):
                print(f'  {number}/{len(points)} puntos')

    if failures:
        print(f'Aviso: {len(failures)} puntos no se pudieron recuperar; se omiten del mapa.')
    if not results:
        raise RuntimeError(f'No se obtuvo ningún punto para {name}')
    field = pd.concat(results, ignore_index=True)
    field.to_pickle(cache_file)
    return field

temperature = download_field(mode=8, name='temperatura')
precipitation = download_field(mode=10, name='precipitacion')
print('Horas comunes:', len(set(temperature.time) & set(precipitation.time)))

In [ ]:
def regular_grid(data, timestamp):
    field = data.loc[data.time == timestamp, ['lat', 'lon', 'value']]
    lat = np.sort(field.lat.unique())
    lon = np.sort(field.lon.unique())
    z = field.pivot(index='lat', columns='lon', values='value').reindex(index=lat, columns=lon).to_numpy()
    return lon, lat, z

def overlay_image(data, timestamp, cmap, levels):
    lon, lat, z = regular_grid(data, timestamp)
    figure, axis = plt.subplots(figsize=(8, 6), dpi=135)
    figure.patch.set_alpha(0)
    axis.set_facecolor((0, 0, 0, 0))
    axis.contourf(lon, lat, z, levels=levels, cmap=cmap, alpha=0.60, extend='both', antialiased=True)
    lines = axis.contour(lon, lat, z, levels=levels, colors='#202020', linewidths=0.65, alpha=0.82)
    axis.clabel(lines, inline=True, fontsize=7, fmt=lambda value: f'{value:g}')
    axis.set_xlim(WEST, EAST)
    axis.set_ylim(SOUTH, NORTH)
    axis.axis('off')
    figure.subplots_adjust(left=0, right=1, bottom=0, top=1)
    buffer = BytesIO()
    figure.savefig(buffer, format='png', transparent=True, dpi=135, pad_inches=0)
    plt.close(figure)
    encoded = base64.b64encode(buffer.getvalue()).decode('ascii')
    return f'data:image/png;base64,{encoded}'

TOPO_BASE = {
    'sourcetype': 'raster',
    'source': ['https://a.tile.opentopomap.org/{z}/{x}/{y}.png'],
    'sourceattribution': '© OpenStreetMap contributors, SRTM | OpenTopoMap (CC-BY-SA)',
    'opacity': 0.48,
    'below': 'traces',
}

def weather_layer(image):
    return {
        'sourcetype': 'image', 'source': image,
        'coordinates': [[WEST, NORTH], [EAST, NORTH], [EAST, SOUTH], [WEST, SOUTH]],
        'below': 'traces', 'opacity': 1,
    }

def map_with_slider(data, title, cmap, step, lower=None, upper=None):
    timestamps = sorted(data.time.unique())
    values = data.value.dropna()
    lower = np.floor(values.quantile(0.02) / step) * step if lower is None else lower
    upper = np.ceil(values.quantile(0.98) / step) * step if upper is None else upper
    if upper <= lower:
        upper = lower + step
    levels = np.arange(lower, upper + step * 1.01, step)
    images = {timestamp: overlay_image(data, timestamp, cmap, levels) for timestamp in timestamps}
    steps = [
        {
            'label': pd.Timestamp(timestamp).strftime('%d %b %H:%M'),
            'method': 'relayout',
            'args': [{'mapbox.layers': [TOPO_BASE, weather_layer(images[timestamp])]}],
        }
        for timestamp in timestamps
    ]
    figure = go.Figure(go.Scattermapbox(
        lat=[CHAMARTIN_LAT], lon=[CHAMARTIN_LON], mode='markers+text',
        text=['Chamartín'], textposition='top right', hoverinfo='skip',
        marker={'size': 8, 'color': '#111111'}, textfont={'color': '#111111', 'size': 12},
        showlegend=False,
    ))
    figure.update_layout(
        title={'text': title, 'x': 0.5}, height=760,
        mapbox={
            'style': 'white-bg', 'center': {'lat': CHAMARTIN_LAT, 'lon': CHAMARTIN_LON},
            'zoom': 8.95, 'layers': [TOPO_BASE, weather_layer(images[timestamps[0]])],
        },
        sliders=[{'active': 0, 'currentvalue': {'prefix': 'Hora local: '}, 'pad': {'t': 45}, 'steps': steps}],
        margin={'l': 0, 'r': 0, 't': 75, 'b': 20},
        paper_bgcolor='white',
    )
    return figure


In [ ]:
temperature_map = map_with_slider(
    temperature, 'AROME · Temperatura a 2 m · entorno de Madrid',
    cmap='RdYlBu_r', step=1
)
temperature_map.show(config={'scrollZoom': True})

In [ ]:
# Precipitación prevista por hora, en milímetros.
precipitation_map = map_with_slider(
    precipitation, 'AROME · Precipitación horaria · entorno de Madrid',
    cmap='PuBuGn', step=0.5, lower=0
)
precipitation_map.show(config={'scrollZoom': True})

## Notas de uso

- El área no sigue límites administrativos: es un cuadrado de ~10.000 km² centrado en Chamartín.
- El fondo es OpenTopoMap (datos de OpenStreetMap y relieve SRTM) con opacidad reducida; puedes acercar, alejar y arrastrar el mapa.
- Las isolíneas y las superficies de color se regeneran para cada hora. Las etiquetas de las líneas muestran el valor de la variable.
- `GRID_STEP = 0.05` equivale a unos 5 km. Reducirlo mejora el detalle, pero incrementa las consultas a Meteociel.